# IDR Prediction — Inference with Pre-computed Features

**Per-residue intrinsically disordered region (IDR) prediction** using a trained LC1 model (bilinear cross-attention) that combines ESM-C protein language model embeddings with Boltz-2 structure features.

This notebook uses **pre-computed embeddings** — no GPU-heavy pLM or structure prediction is needed. It runs on CPU or GPU.

## Pipeline

```
Pre-computed ESM-C embeddings (960d) ─→ ┐
                                         ├─ LC1 Model ─→ P(disorder) per residue
Pre-computed Boltz-2 features (771d) ──→ ┘
```

## Modes

- **Example mode**: Predict from 97 bundled example proteins (CAID3-PDB dataset)
- **Custom mode**: Bring your own pre-computed embeddings + Boltz-2 features

> **GPU not required** — inference runs fine on CPU for moderate-length proteins. Enable GPU for faster batch inference on many/long proteins.

---
## 1. Setup

In [ ]:
#@title 1.1 Clone repository and install dependencies
import os

# TODO: Replace with your actual GitHub repository URL
REPO_URL = "https://github.com/YOUR_USERNAME/idr-prediction-inference.git"  #@param {type:"string"}
REPO_DIR = "/content/idr-prediction"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already cloned at {REPO_DIR}")

# Install minimal dependencies
!pip install -q torch numpy biopython pyyaml

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
#@title 1.2 Verify imports
import sys
sys.path.insert(0, REPO_DIR)

import json
import numpy as np
import torch

from src.models.screening_model import ScreeningModel
from src.data.boltz_loader import BoltzFeatureLoader
from src.data.collate import build_sample, collate_fn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("All imports OK.")

---
## 2. Input Data

### Option A: Use bundled example data (97 proteins from CAID3-PDB)
Simply run the next cell — no uploads needed.

### Option B: Use your own data
You need three things:
1. **FASTA file** — standard format with protein IDs as headers
2. **Pre-computed ESM-C embeddings** — `.pt` file mapping `{protein_id: tensor[L, 960]}`
3. **Boltz-2 predictions directory** — containing per-protein subdirectories with `embeddings/trunk_s.npz`, `embeddings/conf_s.npz`, `plddt_*_model_0.npz`, `pae_*_model_0.npz`, `pde_*_model_0.npz`

Upload via the Colab file browser (left sidebar) or mount Google Drive.

In [ ]:
#@title 2.1 Configure input paths
#@markdown **Option A** — Use bundled example data (default):
USE_EXAMPLE_DATA = True  #@param {type:"boolean"}

#@markdown **Option B** — Custom paths (fill in if USE_EXAMPLE_DATA is False):
CUSTOM_FASTA = ""  #@param {type:"string"}
CUSTOM_EMBEDDINGS = ""  #@param {type:"string"}
CUSTOM_BOLTZ_DIR = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Optional**: predict only specific protein IDs (comma-separated, leave empty for all):
PROTEIN_IDS = ""  #@param {type:"string"}

if USE_EXAMPLE_DATA:
    FASTA_PATH = os.path.join(REPO_DIR, "data", "example_sequences.fasta")
    EMBEDDINGS_PATH = os.path.join(REPO_DIR, "data", "embeddings", "example_esmc_300m.pt")
    BOLTZ_DIR = os.path.join(REPO_DIR, "data", "boltz2_predictions", "predictions")
else:
    FASTA_PATH = CUSTOM_FASTA
    EMBEDDINGS_PATH = CUSTOM_EMBEDDINGS
    BOLTZ_DIR = CUSTOM_BOLTZ_DIR

# Parse protein IDs filter
protein_id_filter = None
if PROTEIN_IDS.strip():
    protein_id_filter = [pid.strip() for pid in PROTEIN_IDS.split(",")]
    print(f"Filtering to {len(protein_id_filter)} protein(s): {protein_id_filter}")

print(f"FASTA:      {FASTA_PATH}")
print(f"Embeddings: {EMBEDDINGS_PATH}")
print(f"Boltz dir:  {BOLTZ_DIR}")

In [ ]:
#@title 2.2 Load sequences and features
# Parse FASTA
sequences = {}
current_id = None
current_seq = []
with open(FASTA_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if current_id is not None:
                sequences[current_id] = "".join(current_seq)
            current_id = line[1:].split()[0]
            current_seq = []
        else:
            current_seq.append(line)
    if current_id is not None:
        sequences[current_id] = "".join(current_seq)

# Apply protein ID filter
if protein_id_filter:
    sequences = {pid: seq for pid, seq in sequences.items() if pid in protein_id_filter}

print(f"Loaded {len(sequences)} sequence(s) from FASTA")

# Load pre-computed embeddings
embeddings = torch.load(EMBEDDINGS_PATH, map_location="cpu", weights_only=False)
print(f"Loaded {len(embeddings)} embedding(s)")

# Build Boltz loader
boltz_loader = BoltzFeatureLoader(prediction_dirs=[BOLTZ_DIR])

# Build samples (skip proteins missing features)
samples = []
skipped = []
for pid in sequences:
    if pid not in embeddings:
        skipped.append((pid, "no embedding"))
        continue
    if not boltz_loader.has_features(pid):
        skipped.append((pid, "no Boltz-2 features"))
        continue
    sample = build_sample(pid, embeddings[pid], boltz_loader)
    samples.append(sample)

print(f"\nValid samples: {len(samples)}")
if skipped:
    print(f"Skipped: {len(skipped)}")
    for pid, reason in skipped[:5]:
        print(f"  {pid}: {reason}")
    if len(skipped) > 5:
        print(f"  ... and {len(skipped) - 5} more")

---
## 3. Load Model and Run Inference

In [ ]:
#@title 3.1 Load trained model
CHECKPOINT_PATH = os.path.join(REPO_DIR, "weights", "best_model.pt")
METRICS_PATH = os.path.join(REPO_DIR, "weights", "eval_test_metrics.json")

# Load checkpoint (contains model weights + config)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
config = checkpoint["config"]

# Build model
model = ScreeningModel(config)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: LC1 + D-SB fusion ({n_params:,} params)")

# Load optimal threshold
if os.path.exists(METRICS_PATH):
    with open(METRICS_PATH) as f:
        metrics = json.load(f)
    threshold = metrics["optimal_threshold"]
    print(f"Threshold: {threshold:.4f} (AUROC={metrics['auroc']:.4f}, F-max={metrics['fmax']:.4f})")
else:
    threshold = 0.5
    print(f"Using default threshold: {threshold}")

In [ ]:
#@title 3.2 Run inference
BATCH_SIZE = 8  #@param {type:"integer"}

probabilities = {}

for i in range(0, len(samples), BATCH_SIZE):
    batch_samples = samples[i : i + BATCH_SIZE]
    batch = collate_fn(batch_samples)

    batch_device = {
        k: v.to(device) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

    with torch.no_grad():
        if device == "cuda":
            with torch.amp.autocast("cuda"):
                output = model(batch_device)
        else:
            output = model(batch_device)

    logits = output["logits"].squeeze(-1)
    probs = torch.sigmoid(logits).cpu().numpy()
    lengths = batch["lengths"].numpy()

    for j, pid in enumerate(batch["protein_ids"]):
        L = int(lengths[j])
        probabilities[pid] = probs[j, :L]

# Summary table
print(f"Predicted {len(probabilities)} protein(s) (threshold={threshold:.4f})")
print()
print(f"{'Protein':<15} {'Length':>6} {'Disordered%':>12} {'IDR residues':>13}")
print("-" * 50)
for pid, prob_arr in probabilities.items():
    n_dis = int((prob_arr >= threshold).sum())
    frac = float((prob_arr >= threshold).mean()) * 100
    print(f"{pid:<15} {len(prob_arr):>6} {frac:>11.1f}% {n_dis:>13}")

---
## 4. Visualization

Plot per-residue disorder probability profiles. Disordered regions (above threshold) are highlighted in red.

In [ ]:
#@title 4.1 Select proteins to plot
#@markdown Leave empty to plot the first 10 proteins, or specify IDs (comma-separated):
PLOT_IDS = ""  #@param {type:"string"}

if PLOT_IDS.strip():
    plot_pids = [pid.strip() for pid in PLOT_IDS.split(",") if pid.strip() in probabilities]
else:
    plot_pids = list(probabilities.keys())[:10]

print(f"Plotting {len(plot_pids)} protein(s): {plot_pids}")

In [ ]:
#@title 4.2 Plot disorder profiles
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Helvetica Neue", "Arial", "DejaVu Sans"]
plt.rcParams["font.size"] = 8
plt.rcParams["axes.linewidth"] = 0.5

n_proteins = len(plot_pids)
fig_height = max(2.5, 2.2 * n_proteins)
fig, axes = plt.subplots(
    n_proteins, 1,
    figsize=(10, fig_height),
    squeeze=False,
    sharex=False,
)

for idx, pid in enumerate(plot_pids):
    ax = axes[idx, 0]
    prob_arr = probabilities[pid]
    positions = np.arange(1, len(prob_arr) + 1)

    # Fill + line for probability
    ax.fill_between(positions, prob_arr, alpha=0.25, color="#2196F3", linewidth=0)
    ax.plot(positions, prob_arr, color="#1565C0", linewidth=0.8)

    # Threshold line
    ax.axhline(
        y=threshold, color="#E53935", linestyle="--",
        linewidth=0.5, alpha=0.7, label=f"threshold={threshold:.3f}",
    )

    # Highlight disordered regions
    disordered = prob_arr >= threshold
    diff = np.diff(np.concatenate(([0], disordered.astype(int), [0])))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    for s, e in zip(starts, ends):
        ax.axvspan(s + 1, e, alpha=0.08, color="#E53935")

    ax.set_ylabel("P(disorder)", fontsize=7)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlim(1, len(prob_arr))

    frac = float(disordered.mean()) * 100
    ax.set_title(
        f"{pid}  ({len(prob_arr)} aa, {frac:.1f}% disordered)",
        fontsize=8, fontweight="bold", loc="left",
    )
    ax.tick_params(axis="both", which="both", labelsize=7, width=0.5)
    ax.legend(fontsize=6, loc="upper right", framealpha=0.7)

axes[-1, 0].set_xlabel("Residue position", fontsize=7)
plt.tight_layout()
plt.show()

---
## 5. Export Results

In [ ]:
#@title 5.1 Save predictions to JSON and TSV
OUTPUT_DIR = "/content/predictions"  #@param {type:"string"}
os.makedirs(OUTPUT_DIR, exist_ok=True)

# JSON — per-protein summary with probabilities
results = {}
for pid, prob_arr in probabilities.items():
    disordered = (prob_arr >= threshold).astype(int)
    results[pid] = {
        "length": len(prob_arr),
        "threshold": threshold,
        "fraction_disordered": float(disordered.mean()),
        "probabilities": prob_arr.tolist(),
        "predictions": disordered.tolist(),
    }

json_path = os.path.join(OUTPUT_DIR, "predictions.json")
with open(json_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"JSON saved: {json_path}")

# TSV — per-residue predictions
tsv_path = os.path.join(OUTPUT_DIR, "predictions.tsv")
with open(tsv_path, "w") as f:
    f.write("protein_id\tposition\tamino_acid\tprobability\tprediction\n")
    for pid, prob_arr in probabilities.items():
        seq = sequences.get(pid, "X" * len(prob_arr))
        for pos, (aa, prob) in enumerate(zip(seq, prob_arr), start=1):
            pred = 1 if prob >= threshold else 0
            f.write(f"{pid}\t{pos}\t{aa}\t{prob:.4f}\t{pred}\n")
print(f"TSV saved: {tsv_path}")

In [ ]:
#@title 5.2 Download results (Colab only)
#@markdown Run this cell to download prediction files to your local machine.
DOWNLOAD = False  #@param {type:"boolean"}

if DOWNLOAD:
    try:
        from google.colab import files
        files.download(json_path)
        files.download(tsv_path)
        print("Download started.")
    except ImportError:
        print("Not running in Colab — files are saved locally at:")
        print(f"  {json_path}")
        print(f"  {tsv_path}")
else:
    print(f"Set DOWNLOAD=True to download, or find files at:")
    print(f"  {json_path}")
    print(f"  {tsv_path}")

---
## 6. Explore Individual Proteins

Use this section to interactively inspect a single protein's disorder profile with amino acid annotations.

In [ ]:
#@title 6.1 Detailed single-protein view
INSPECT_ID = "DP03748"  #@param {type:"string"}

if INSPECT_ID not in probabilities:
    available = list(probabilities.keys())[:10]
    print(f"'{INSPECT_ID}' not found. Available: {available}")
else:
    prob_arr = probabilities[INSPECT_ID]
    seq = sequences.get(INSPECT_ID, "X" * len(prob_arr))
    disordered = prob_arr >= threshold

    print(f"Protein: {INSPECT_ID}")
    print(f"Length:  {len(prob_arr)} residues")
    print(f"IDR:     {int(disordered.sum())} residues ({float(disordered.mean())*100:.1f}%)")
    print()

    # Find disordered regions
    diff = np.diff(np.concatenate(([0], disordered.astype(int), [0])))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]

    if len(starts) > 0:
        print("Disordered regions:")
        for s, e in zip(starts, ends):
            region_seq = seq[s:e]
            mean_prob = float(prob_arr[s:e].mean())
            print(f"  {s+1}-{e}: {region_seq[:30]}{'...' if len(region_seq) > 30 else ''} "
                  f"(mean P={mean_prob:.3f})")
    else:
        print("No disordered regions detected.")

    # Detailed plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4), gridspec_kw={"height_ratios": [3, 1]})

    positions = np.arange(1, len(prob_arr) + 1)

    # Top: probability profile
    ax1.fill_between(positions, prob_arr, alpha=0.25, color="#2196F3")
    ax1.plot(positions, prob_arr, color="#1565C0", linewidth=0.8)
    ax1.axhline(y=threshold, color="#E53935", linestyle="--", linewidth=0.5, alpha=0.7)
    for s, e in zip(starts, ends):
        ax1.axvspan(s + 1, e, alpha=0.08, color="#E53935")
    ax1.set_ylabel("P(disorder)", fontsize=7)
    ax1.set_ylim(-0.02, 1.02)
    ax1.set_xlim(1, len(prob_arr))
    ax1.set_title(f"{INSPECT_ID}", fontsize=9, fontweight="bold", loc="left")
    ax1.tick_params(labelsize=7, width=0.5)

    # Bottom: binary prediction as heatmap
    pred_binary = disordered.astype(float).reshape(1, -1)
    ax2.imshow(pred_binary, aspect="auto", cmap="RdBu_r", vmin=0, vmax=1,
               extent=[0.5, len(prob_arr) + 0.5, 0, 1])
    ax2.set_yticks([])
    ax2.set_xlabel("Residue position", fontsize=7)
    ax2.set_ylabel("Pred", fontsize=7)
    ax2.tick_params(labelsize=7, width=0.5)

    plt.tight_layout()
    plt.show()